# FB15K-237 — seed 2029: early stopping
Maximum 50 epochs; monitor combined validation retrieval MRR; patience=5; min_delta=1e-5.
The best checkpoint always tracks the highest validation MRR. Early-stopping state is saved for resume.
Output directory: `fb15k237_clean_kgc_to_retrieval_50ep_seed2029_v1`. Start with RESUME_TRAINING=False; set True to resume this run.


# AdaMF training & validation — workflow 04a, FB15K-237-IMG, 50 epoch
Theo workflow `04a_gat_training_validation (1).ipynb`: frozen CLIP → train queries + hard negatives →
query-image cross entropy → mỗi epoch validation retrieval → chọn alpha và checkpoint bằng validation MRR.

Vẫn dùng **FB15K-237-IMG** theo yêu cầu đang thực hiện; 04a gốc dùng **Visual Genome split_v2**, nên không
dùng các file/counts của VG cho FB15K. Để so sánh chỉ khác thuật toán, GAT trên FB15K cũng cần dùng cùng
cache/query IDs/top-k và protocol do notebook này xuất. Không thể khẳng định hai dataset có cùng split.

AdaMF thích nghi cho retrieval, `mu=0` (không MAT): head/tail dùng attention fusion + RotatE score;
direct dùng text projection so với fused entity embeddings. Không dùng target ID làm đầu vào scoring.
Loss là query-image contrastive giống 04a, khác notebook KGC trước; đây không phải tái lập nguyên bản loss AdaMF-MAT.

Giữ 50 epoch theo yêu cầu; seed **2026** giống 04a. Cấu hình và workflow train/validation bám 04a:
417.700 train queries tối đa, 50 hard negatives, batch 8, AdamW lr=1e-4/weight_decay=1e-4,
validation mỗi epoch, z-score từng query và alpha grid 0..1. Không đánh giá hoặc tuning trên test.
FB15K có 29.088 validation queries nên sử dụng toàn bộ, không tạo thêm query để bằng số lượng Visual Genome.


## 1. Drive, GPU và cấu hình


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 3, globals())


## 2. Source AdaMF và artifact manifest


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 5, globals())


## 3. Kiểm tra adapter và features canonical
Giữ các kiểm tra mapping/split/mask của bản trước. Sau đó visual entity features sẽ được **pool lại chỉ từ ảnh train**
trong runtime cache để không đưa ảnh validation/test vào lớp chiếu khi học; canonical Drive không bị sửa.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 7, globals())


## 4. Retrieval tables và entity visual features chỉ từ train
Image-level features đóng băng. Candidate images được chọn theo đúng split canonical.
Cache được khóa bằng hashes nguồn và protocol; đổi tham số tạo cache thì dùng namespace mới tự động.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 9, globals())


## 5. Cache CLIP text features cho train/validation
Dùng encoder được metadata ảnh khai báo, đóng băng toàn bộ. Không encode query test.
Metadata gốc chưa khóa commit encoder, nên lần đầu resolve commit hiện tại và khóa nó trong cache/run.
Các mô hình so sánh cần dùng chính cache này để có CLIP baseline giống nhau.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 11, globals())


## 6. CLIP hard negatives và validation top-k cache
Hard negatives lấy duy nhất từ train pool, loại **tất cả** positive của query.
Validation giữ top-k CLIP và full-pool best-positive rank. Không có positive trong top-k thì giữ rank baseline.
Nếu có nhiều positive trong top-k, dùng positive có rank tốt nhất sau rerank.
Giống 04a, rank = 1 + số điểm lớn hơn điểm positive; ties dùng optimistic rank.
Riêng chọn ảnh ở ranh giới top-k dùng feature_row để cache ổn định.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 13, globals())


## 7. AdaMF retrieval scorer
Head/tail: dùng anchor và relation của query để tính `F(h,r,candidate_entity)` hoặc `F(candidate_entity,r,t)`.
Direct: cosine giữa text projection và AdaMF fused entity vector. Điểm ảnh là max qua các linked entities.
Không đưa `target_entity_index` vào scorer. Features đầu vào frozen, modality mask trước softmax.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 15, globals())


## 8. Validation — CLIP, CLIP + AdaMF và alpha grid
Như 04a: sample-standard-deviation z-score từng query và alpha 0..1 bước 0.1.
`alpha_clip` nhân CLIP; `alpha_kg=1-alpha_clip` tương ứng ký hiệu trong handoff FB15K.
Chọn theo combined validation MRR; đồng thời log direct/indirect-head/indirect-tail.
Recall@K dùng ít nhất một relevant image trong top K, giống hit-rate theo best-positive-rank của 04a.
Đây không phải fraction-recall của tất cả positive images; định nghĩa được xuất trong protocol.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 17, globals())


## 9. Khóa identity và train 50 epoch với validation mỗi epoch
Train batch: 1 positive được sample từ toàn bộ relevance train + 50 CLIP hard negatives, query batch 8 như 04a.
Checkpoint `best_adamf.pt` chọn validation MRR; `last_checkpoint.pt` phục vụ resume.
Mỗi 500 batch hoặc 30 phút lưu checkpoint. Lưu trạng thái trước validation để có thể chạy lại validation nếu bị ngắt.
Timer dừng phiên trước thời hạn và lưu tiến độ; GPU phải còn đủ thời gian cho một lượt validation.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 19, globals())


## 10. Biểu đồ loss, validation MRR và bảng CLIP vs AdaMF


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 21, globals())


## Outputs
Checkpoint và metric train/validation nằm trong `RUN/seed_<seed>/`. Cache CLIP dùng chung, có kiểm tra identity.
Giữ nguyên cấu hình và đặt `RESUME_TRAINING=True` để tiếp tục checkpoint cũ.


## 11. Đánh giá cuối cùng trên tập test
Chỉ chạy sau khi có `complete.json` (đủ epoch hoặc early stopping). Dùng `best_adamf.pt` và alpha đã chọn trên validation; không train hoặc chọn siêu tham số trên test.
Báo cáo CLIP và CLIP + AdaMF: MRR, Recall@1/5/10/50; tách direct, indirect, head và tail.
Kết quả lưu tại `RUN/seed_<seed>/test_locked_<timestamp>/`: `test_metrics.csv`, `test_query_ranks.csv`, `test_protocol.json`.


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('04b2_adamf_fb15k237_seed2029', 24, globals())
